# optimizing the model

This notebook cell adapts `deep_correlation_model.py` into an interactive notebook cell. It trains an MLP to jointly maximize Pearson correlation and minimize MSE using a combined loss.

The cell below contains: explanation, model definition, training loop, metrics, and plots.

In [ ]:
%matplotlib inline
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Seed for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)

# Tiny dataset class for notebook
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32).reshape(-1,1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Model: MLP with BatchNorm, GELU and dropout (good defaults)
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128,64), dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev,1))
        self.net = nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)

# Pearson correlation (differentiable)
def pearson_corr(pred, target, eps=1e-8):
    p = pred.view(-1)
    t = target.view(-1)
    p = p - p.mean()
    t = t - t.mean()
    num = (p*t).sum()
    den = torch.sqrt((p*p).sum() * (t*t).sum()).clamp(min=eps)
    return (num/den).item() if den.item()!=0 else 0.0

# Combined loss: alpha*MSE + beta*(1-corr)
class CombinedLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.mse = nn.MSELoss()
        self.alpha = alpha
        self.beta = beta
    def forward(self, pred, target):
        mse = self.mse(pred, target)
        # compute pearson on CPU-safe tensor to avoid NaNs in gradient path for metric reporting
        # for loss we compute correlation as a differentiable tensor
        p = pred.view(-1)
        t = target.view(-1)
        p = p - p.mean()
        t = t - t.mean()
        num = (p*t).sum()
        den = torch.sqrt((p*p).sum() * (t*t).sum()).clamp(min=1e-8)
        corr = num/den
        corr_loss = 1.0 - corr
        return self.alpha*mse + self.beta*corr_loss, mse.detach(), corr.detach()

# Prepare a quick synthetic demo dataset (or load your data here)
N = 5000
D = 20
X = np.random.randn(N,D)
w = np.linspace(1.0, 0.2, D)
y = X.dot(w) + 0.4*np.sin(X[:,0]) + 0.1*np.random.randn(N)
# standardize features
X = (X - X.mean(axis=0)) / (X.std(axis=0)+1e-9)
y = (y - y.mean())/ (y.std()+1e-9)

# split
val_frac = 0.15
idx = np.arange(N)
np.random.shuffle(idx)
val_n = int(N*val_frac)
val_idx = idx[:val_n]
train_idx = idx[val_n:]
X_train = X[train_idx]
y_train = y[train_idx]
X_val = X[val_idx]
y_val = y[val_idx]

train_ds = TabularDataset(X_train, y_train)
val_ds = TabularDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(in_dim=D, hidden=(256,128,64), dropout=0.15).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
loss_fn = CombinedLoss(alpha=1.0, beta=1.0)

# Training loop with simple logging and plots
epochs = 60
history = {'train_loss':[], 'val_loss':[], 'train_mse':[], 'val_mse':[], 'train_corr':[], 'val_corr':[]}
for ep in range(1, epochs+1):
    model.train()
    tloss = 0.0; tmse = 0.0; tcorr = 0.0; n=0
    for Xb, yb in train_loader:
        Xb = Xb.to(device); yb = yb.to(device)
        opt.zero_grad()
        preds = model(Xb)
        loss, mse_val, corr_val = loss_fn(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step()
        bs = Xb.size(0)
        tloss += loss.item()*bs; tmse += mse_val.item()*bs; tcorr += corr_val.item()*bs; n += bs
    sched.step()
    train_loss = tloss/n; train_mse = tmse/n; train_corr = tcorr/n

    # validation
    model.eval()
    vloss = 0.0; vmse = 0.0; vcorr = 0.0; nnv=0
    preds_all = []
    targ_all = []
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb = Xb.to(device); yb = yb.to(device)
            preds = model(Xb)
            loss, mse_val, corr_val = loss_fn(preds, yb)
            bs = Xb.size(0)
            vloss += loss.item()*bs; vmse += mse_val.item()*bs; vcorr += corr_val.item()*bs; nnv += bs
            preds_all.append(preds.cpu().numpy())
            targ_all.append(yb.cpu().numpy())
    val_loss = vloss/nnv; val_mse = vmse/nnv; val_corr = vcorr/nnv

    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_mse'].append(train_mse); history['val_mse'].append(val_mse)
    history['train_corr'].append(train_corr); history['val_corr'].append(val_corr)

    if ep % 5 == 0 or ep==1:
        print(f'Epoch {ep:03d}: tr_loss={train_loss:.5f} val_loss={val_loss:.5f} tr_corr={train_corr:.4f} val_corr={val_corr:.4f}')

# Plot metrics
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='val loss')
plt.legend(); plt.title('Combined loss')

plt.subplot(1,2,2)
plt.plot(history['train_corr'], label='train corr')
plt.plot(history['val_corr'], label='val corr')
plt.legend(); plt.title('Pearson correlation (1 - 1 gives 0 best)')
plt.show()

# scatter preds vs targets on validation set
preds_all = np.vstack(preds_all).ravel()
targ_all = np.vstack(targ_all).ravel()
plt.figure(figsize=(5,5))
plt.scatter(targ_all, preds_all, alpha=0.3, s=10)
mn = min(targ_all.min(), preds_all.min()); mx = max(targ_all.max(), preds_all.max())
plt.plot([mn,mx],[mn,mx], color='k', linestyle='--')
plt.xlabel('target'); plt.ylabel('pred'); plt.title('Val: preds vs targets')
plt.show()

# Print final metrics
print('Final val MSE:', np.mean((preds_all - targ_all)**2))
print('Final val Pearson r:', np.corrcoef(preds_all, targ_all)[0,1])